# User Interface

This Notebook generates a simple user interface using Gradio and implements the fake news model and the bias news model to make predictions on new articles that can be cut and pasted into the user interface.

*Note - You cannot cut and paste into the notebook version of the UI.  You must cut and paste the text into the browser pop-up of the user interface*

In [ ]:
# basic impoorts
import pandas as pd
from pandas import DataFrame
import numpy as np

# machine learning imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn import svm
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression

#user interface
import gradio as gr
import sqlite3
import calendar

from joblib import load

#### Load Functions from Other Notebooks

First we use joblib to load the vectorizers, models, preprocessing fand lemmatization function

In [3]:
bias_vectorizer = load("vec_text.joblib")
bias_model = load("grid_best_estimator_logreg.joblib")

In [ ]:

## load and dump maria's model and vectorizer
## also need to dump and load lemmatizer and preprocessor from Maria's code

#### Define New Functions for UI

Next we define a function to analyze the bias of a given article, a function to analyze the truthfulness of the given article, and a function to tie them together.  

In [4]:
def analyze_bias(user_first, news_text):
    X = bias_vectorizer.transform([news_text])
    pred = bias_model.predict(X)[0]
    prob = bias_model.predict_proba(X).max()

    label = "left-leaning" if pred == 0 else "right-leaning"

    # --- NEW: get top contributing words ---
    feature_names = bias_vectorizer.get_feature_names_out()
    coefs = bias_model.coef_[0]

    indices = X.nonzero()[1]

    contributions = []
    for i in indices:
        value = X[0, i]
        contribution = value * coefs[i]
        contributions.append((feature_names[i], contribution))

    # sort by absolute contribution and take top 5
    top_words = sorted(contributions, key=lambda x: abs(x[1]), reverse=True)[:5]
    top_words = [word for word, _ in top_words]
    # --- end new section ---

    return f"Hi {user_first}, your article has a {label} political bias. I am {prob*100:.1f}% certain. Top words: {', '.join(top_words)}."

In [ ]:
def analyze_fake_news(user_first, news_text):

    clean_news = prepro(news_text)
    
    X = tdfit.transform([clean_news])
    pred = logs_reg.predict(X)[0]
    prob = logs_reg.predict_proba(X).max()

    label = "Fake" if pred == 0 else "Real"

    """
    # --- NEW: get top contributing words ---
    feature_names = tdfit.get_feature_names_out()
    coefs = grid_best_estimator_3.coef_[0]

    indices = X.nonzero()[1]

    contributions = []
    for i in indices:
        value = X[0, i]
        contribution = value * coefs[i]
        contributions.append((feature_names[i], contribution))

    # sort by absolute contribution and take top 5
    top_words = sorted(contributions, key=lambda x: abs(x[1]), reverse=True)[:5]
    top_words = [word for word, _ in top_words]
    # --- end new section ---
"""
   # return f"Hi {user_first}, your article has a {label} political bias. I am {prob*100:.1f}% certain. Top words: {', '.join(top_words)}."
    return (f"Hi {user_first}, your article has been labelled as {label} news. I am {prob*100:.1f}% certain.")

## Gradio Interface

Next we code the gradio interface.  This pops up a browser window that asks you to enter your name and the text you want to analyze.  The app will then preprocess the text and run both the bias model and the fake news model to predict whether the text is real or fake and whether it is left or right leaning.  

In [ ]:
with gr.Blocks(css="""
#header { text-align: center; color: darkblue; font-family: Arial; }
.primary-btn { background-color: #4CAF50; color: white; font-weight: bold; }

.result-text {
    background-color: white;
    padding: 12px;
    border-radius: 10px;
    border-left: 6px solid #b30000;
}

.result-text, .result-text * {
    color: #333333 !important;
    font-size: 18px;
}
""") as app:

    gr.Markdown("# Welcome to the News Detector", elem_id="header")

    with gr.Column(visible=True) as info_screen:

        user_first = gr.Textbox(label="Name", placeholder="Enter your first name")

        news = gr.Textbox(label="News Article Text", placeholder="Enter text here")

        submit_txt_btn = gr.Button("Submit Text for Analysis", elem_classes="primary-btn")

        result = gr.Textbox(label = "Results", elem_classes="result-text")

    submit_txt_btn.click(
        fn=analyze_both,
        inputs=[user_first, news],
        outputs=result
    )

app.launch(inbrowser=True, share=True)